# Assemble Highlight Reels

Find and assemble relevant clips from a video collection based on a theme, topic, or criteria, producing a list of clip references ready for your editing pipeline.

In [ ]:
import json
import os

import requests

# Configuration
API_KEY = os.environ.get("TWELVELABS_API_KEY", "YOUR_API_KEY")
BASE_URL = "https://api.twelvelabs.io/v1.3"
HEADERS = {"x-api-key": API_KEY, "Content-Type": "application/json"}

# Replace with your knowledge store ID
STORE_ID = "your_knowledge_store_id"

## Helper Functions

Utility functions for parsing Jockey API responses.

In [ ]:
def parse_response(result: dict) -> str | dict:
    """Extract text content from a Jockey API response."""
    for output in result["output"]:
        if output["type"] == "message":
            for content in output["content"]:
                return content["text"]
    return ""


def parse_json_response(result: dict) -> dict:
    """Extract and parse JSON content from a Jockey API response."""
    text = parse_response(result)
    if isinstance(text, str) and text:
        return json.loads(text)
    return {}

## Clip Assembly Schema

Define the output structure for the highlight reel. Each clip includes a video reference, start/end timestamps, a description, and a relevance reason. The schema also captures the assembly title, estimated total duration, and editorial notes.

In [ ]:
CLIP_SCHEMA = {
    "type": "object",
    "properties": {
        "assembly_title": {"type": "string"},
        "clips": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "video_reference": {"type": "string"},
                    "start_time": {"type": "string"},
                    "end_time": {"type": "string"},
                    "description": {"type": "string"},
                    "relevance_reason": {"type": "string"},
                },
            },
        },
        "total_estimated_duration": {"type": "string"},
        "assembly_notes": {"type": "string"},
    },
}

## Query for Clips

Send a query to the Jockey API with:

1. **Instructions** that set the editorial persona (e.g., "video editor assembling a highlight reel").
2. **A user message** describing what clips you want and the target duration.
3. **Structured output** via `text.format` so the response is typed and parseable.

The `instructions` field shapes how Jockey selects and orders clips -- think of it as directing the editorial style.

In [ ]:
response = requests.post(
    f"{BASE_URL}/responses",
    headers=HEADERS,
    json={
        "model": "jockey1.0",
        "instructions": (
            "You are a video editor assembling a highlight reel. "
            "Select clips that flow well together with good pacing and variety."
        ),
        "input": [
            {
                "type": "message",
                "role": "user",
                "content": (
                    "Find the best clips showing product demos and customer "
                    "reactions. I need a 2-minute highlight reel."
                ),
            }
        ],
        "knowledge_store_id": STORE_ID,
        "text": {"format": {"type": "json_schema", "name": "highlight_reel", "schema": CLIP_SCHEMA}},
    },
)

result = response.json()
session_id = result["session_id"]
assembly = parse_json_response(result)

print(f"Assembly: {assembly['assembly_title']}")
print(f"Duration: {assembly['total_estimated_duration']}")
print(f"Notes: {assembly['assembly_notes']}")
print()
for i, clip in enumerate(assembly["clips"], 1):
    print(f"  {i}. [{clip['start_time']}-{clip['end_time']}] {clip['description']}")
    print(f"     Video: {clip['video_reference']}")
    print(f"     Reason: {clip['relevance_reason']}")

## Example Response

The structured output looks like this:

```json
{
  "assembly_title": "Product Demo Highlights Q1",
  "clips": [
    {
      "video_reference": "marketing_video_03",
      "start_time": "00:01:22",
      "end_time": "00:01:45",
      "description": "Close-up product walkthrough with feature callouts",
      "relevance_reason": "Strong visual demonstration of core feature"
    },
    {
      "video_reference": "customer_interview_07",
      "start_time": "00:03:10",
      "end_time": "00:03:38",
      "description": "Customer describing their positive experience",
      "relevance_reason": "Authentic testimonial with emotional impact"
    }
  ],
  "total_estimated_duration": "1:52",
  "assembly_notes": "Opens with product demo for context, transitions to customer reactions for social proof"
}
```

## Refine with a Follow-Up Turn

Use the `session_id` from the initial query to continue the conversation and refine the clip selection. This is useful for swapping clips, adjusting pacing, or changing the editorial direction.

In [ ]:
response = requests.post(
    f"{BASE_URL}/responses",
    headers=HEADERS,
    json={
        "model": "jockey1.0",
        "session_id": session_id,
        "input": [
            {
                "type": "message",
                "role": "user",
                "content": "Replace the second clip with something more energetic.",
            }
        ],
        "knowledge_store_id": STORE_ID,
        "text": {"format": {"type": "json_schema", "name": "highlight_reel", "schema": CLIP_SCHEMA}},
    },
)

refined = parse_json_response(response.json())

print(f"Refined Assembly: {refined.get('assembly_title', 'N/A')}")
for i, clip in enumerate(refined.get("clips", []), 1):
    print(f"  {i}. [{clip['start_time']}-{clip['end_time']}] {clip['description']}")

## Variations

- **Change the instructions** to shift selection style: `"documentary editor"` vs `"social media creator"` vs `"training video producer"`.
- **Change the query** to focus on different content: `"funny moments"`, `"technical deep dives"`, `"executive summaries"`.
- **Add a follow-up turn** to refine: `"Replace the second clip with something more energetic."`

## Next Steps

- [Organize a Video Library](./organize_video_library.ipynb) -- Categorize your collection before assembling reels.
- [Build a Content Agent](./build_content_agent.ipynb) -- Create multi-step agents that produce structured creative output.
- [Assemble Highlight Reels (docs)](../../docs/cookbooks/assemble-highlight-reels.md) -- Full reference documentation.